# 04 — Diagnóstico de trayectorias de 4 apariciones (t, t+1, t+4, t+5)

Diagnóstico de factibilidad de seguir cada hogar en sus 4 apariciones dentro de una cohorte
de rotación "overlapping" (t, t+1, t+4, t+5) -- **no construye ninguna etiqueta de deterioro**
ni entrena nada. Responde: cuántas trayectorias hay, dónde se cortan, cuán selectiva es la
atrición, y qué N da cada diseño candidato de panel dinámico.

Lee `data/02_nucleo/`, `data/03_panel/` y (para `NIVEL_ED`, igual que el notebook 03)
`data/01_armonizado/individual/`. Escribe `data/04_trayectorias/` (Parquet, gitignored) y los
CSV resumen de `data/meta/` descriptos en cada sección. Ver el plan aprobado en
`docs/decisiones_metodologicas.md` D12 para el detalle completo de las reglas usadas acá, y
D13 para la anomalía de datos 2013-2014 que motivó la Ronda 2 de correcciones de este notebook.

**Dos claves distintas**: la rotación 2-2-2 es propiedad de la *vivienda* (`CODUSU+AGLOMERADO`)
-- se usa en `inferir_visita` y en la distribución de visitas. La identidad de *hogar*
(`CODUSU+NRO_HOGAR+AGLOMERADO`, D9) es otra cosa -- se usa para encadenar trayectorias y medir
selectividad.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from perfil_hogar import *
from constantes import *

def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

from armonizacion import trimestres_publicados, es_historico
import panel
import pipeline_meta as pm
import trayectorias as tr

TRIMESTRES: list[tuple[int, int]] | None = None  # subconjunto opcional de ORIGENES t0; None = todos
REHACER: bool = False  # True fuerza reprocesar aunque la partición esté al día

NUCLEO = REPO / "data" / "02_nucleo"
ARMONIZADO = REPO / "data" / "01_armonizado"
PANEL = REPO / "data" / "03_panel"
TRAYECTORIAS = REPO / "data" / "04_trayectorias"
META = REPO / "data" / "meta"
(TRAYECTORIAS / "trayectorias").mkdir(parents=True, exist_ok=True)

TODOS_LOS_TRIMESTRES = trimestres_publicados()
trimestres_origen = TRIMESTRES if TRIMESTRES is not None else TODOS_LOS_TRIMESTRES
ES_CORRIDA_PARCIAL = TRIMESTRES is not None
COMMIT_GIT = pm.obtener_commit_git(REPO)
t0_proceso = time.time()
print(f"{len(trimestres_origen)} trimestres de origen a procesar" +
      (" -- CORRIDA PARCIAL (TRIMESTRES acotado)" if ES_CORRIDA_PARCIAL else ""))

## Funciones de categorización -- duplicadas del notebook 03

`cargar_hogar`/`cargar_individual` se importan de `pipeline_meta` (uso general,
compartido con 03/05/06), no se duplican. El resto de las funciones de esta celda sí
sigue duplicado tal cual de `03_exploracion_descriptiva.ipynb`: no existe un módulo
compartido para ellas (viven como celdas del notebook 03, no en `src/`), y para usar
las mismas categorías sin tocar un archivo versionado existente (regla de cambios
aditivos) se duplican acá tal cual.

In [ ]:
COLUMNAS_HOGAR = [
    "CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "DECCFR", "PONDIH", "IPCF",
    "II1", "II2", "II5", "II5_1", "IV1", "IV3", "IV8", "IV10", "IV12_1", "IV12_2", "IV12_3",
    "V13", "V14", "V15", "V16", "V17", "ingreso_no_declarado",
]
COLUMNAS_INDIVIDUAL = [
    "CODUSU", "NRO_HOGAR", "COMPONENTE", "ANO4", "TRIMESTRE",
    "CH03", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H", "PONDERA",
]

def calcular_info_jefe(individual: pd.DataFrame) -> pd.DataFrame:
    jefe = individual[individual["CH03"] == 1]
    conteo = jefe.groupby(["CODUSU", "NRO_HOGAR"]).size()
    jefe_unico = jefe.drop_duplicates(["CODUSU", "NRO_HOGAR"]).set_index(["CODUSU", "NRO_HOGAR"])
    estado = pd.to_numeric(jefe_unico["ESTADO"], errors="coerce")
    condicion = pd.Series(pd.array([pd.NA] * len(jefe_unico), dtype="object"), index=jefe_unico.index)
    condicion[estado == 1] = "ocupado"
    condicion[estado == 2] = "desocupado"
    condicion[estado == 3] = "inactivo"
    resultado = pd.DataFrame({
        "condicion_actividad_jefe": condicion,
        "pondera_jefe": jefe_unico["PONDERA"],
        "nivel_ed_jefe": jefe_unico["NIVEL_ED"],
        "sexo_jefe": jefe_unico["CH04"],
    }).reset_index()
    resultado = resultado.merge(conteo.rename("n_jefe").reset_index(), on=["CODUSU", "NRO_HOGAR"], how="left")
    resultado["jefe_ambiguo"] = resultado["n_jefe"].fillna(0).ne(1)
    resultado = resultado.drop(columns=["n_jefe"])
    resultado.loc[resultado["jefe_ambiguo"], ["condicion_actividad_jefe", "sexo_jefe"]] = pd.NA
    return resultado

## Paso 0 — claves por trimestre

`CODUSU+AGLOMERADO` (vivienda) y `CODUSU+NRO_HOGAR+AGLOMERADO` (hogar) de cada trimestre
publicado, insumo común para `inferir_visita` y para el patrón de apariciones. Trimestres sin
partición en `02_nucleo/hogar` quedan afuera (no debería pasar en una corrida completa, los 86
publicados tienen partición -- ver CLAUDE.md).

In [ ]:
CLAVES_MINIMAS = ["CODUSU", "NRO_HOGAR", "AGLOMERADO"]

claves_vivienda_por_trimestre: dict[tuple[int, int], pd.DataFrame] = {}
claves_hogar_por_trimestre: dict[tuple[int, int], pd.DataFrame] = {}
n_sin_particion = 0
for anio, trimestre in TODOS_LOS_TRIMESTRES:
    p = pm.ruta_particion(NUCLEO, "hogar", anio, trimestre)
    if not p.exists():
        n_sin_particion += 1
        continue
    hogar = pd.read_parquet(p, columns=CLAVES_MINIMAS)
    claves_vivienda_por_trimestre[(anio, trimestre)] = tr.claves_vivienda(hogar)
    claves_hogar_por_trimestre[(anio, trimestre)] = tr.claves_hogar(hogar)

print(f"{len(claves_vivienda_por_trimestre)} trimestres con claves cargadas, "
      f"{n_sin_particion} sin partición en 02_nucleo/hogar")

## Sección 1 — inferir_visita y encadenamiento

### Inferencia de visita (método de hipótesis simétrico, D12)

Para cada trimestre `t`, se arma el diccionario de presencia por offset (`0` más los 8 de
`tr.OFFSETS_VISITA`) resolviendo cada offset con `panel.sumar_trimestres` y buscando en
`claves_vivienda_por_trimestre`. Un offset es `None` (no determinable) si el trimestre destino no
está en el diccionario (fuera de la serie o hueco de publicación) **o** si está en otra era que
`t` (`es_historico` distinto, D7) -- ambos casos se tratan igual, sin distinguir anomalía.

### Matriz de solapamiento de `CODUSU`, 2012T4-2014T4 (D13)

Antes de definir qué offsets tratar como no determinables, mido el % de intersección de `CODUSU` entre cada par de trimestres de esta ventana, en ambas direcciones -- para delimitar exactamente qué comparaciones están rotas en vez de asumir un bloque.

In [ ]:
def _claves_codusu(anio: int, trimestre: int) -> set[str]:
    return set(claves_vivienda_por_trimestre[(anio, trimestre)]["CODUSU"])


_ventana_matriz = [(2012, 4), (2013, 1), (2013, 2), (2013, 3), (2013, 4),
                    (2014, 1), (2014, 2), (2014, 3), (2014, 4)]
_sets_matriz = {q: _claves_codusu(*q) for q in _ventana_matriz}
_etiquetas_matriz = [f"{a}T{t}" for a, t in _ventana_matriz]
_filas_matriz = []
for _qa, _la in zip(_ventana_matriz, _etiquetas_matriz):
    _fila = {"origen": _la}
    for _qb, _lb in zip(_ventana_matriz, _etiquetas_matriz):
        if _qa == _qb:
            _fila[_lb] = None
            continue
        _fila[_lb] = round(len(_sets_matriz[_qa] & _sets_matriz[_qb]) / len(_sets_matriz[_qa]) * 100, 1)
    _filas_matriz.append(_fila)
matriz_codusu = pd.DataFrame(_filas_matriz).set_index("origen")
display(matriz_codusu)

**Patrón**: la ruptura es direccional entre dos conjuntos chicos, no un bloque. Cualquier par con un lado en `{2013T1,2013T2,2013T3}` y el otro en `{2014T1,2014T2,2014T3}` da 0% o la mitad de lo esperado, en cualquier distancia de offset -- mientras que `2013T4` compara normal contra los 4 trimestres de 2014, `2014T1-T3` comparan normal entre sí y hacia adelante (`2014T1→2015T1` normal), y `2012T4-2013T4` comparan normal entre sí en todos los gaps. `_presencia_offsets` trata como no determinable cualquier offset cuyo `(origen, destino)` cruce esos dos conjuntos, con el mismo criterio que ya usa para el cruce histórico/regular de D7.

In [ ]:
RUPTURA_D13_ORIGEN = {(2013, 1), (2013, 2), (2013, 3)}
RUPTURA_D13_DESTINO = {(2014, 1), (2014, 2), (2014, 3)}
VISITA_AFECTADA_D13 = RUPTURA_D13_ORIGEN | RUPTURA_D13_DESTINO  # trimestres cuya propia inferir_visita toca la ruptura


def _cruza_ruptura_d13(origen: tuple[int, int], destino: tuple[int, int]) -> bool:
    return ((origen in RUPTURA_D13_ORIGEN and destino in RUPTURA_D13_DESTINO)
            or (origen in RUPTURA_D13_DESTINO and destino in RUPTURA_D13_ORIGEN))


def _presencia_offsets(anio_t: int, trimestre_t: int) -> dict[int, pd.DataFrame | None]:
    historico_t = es_historico(anio_t, trimestre_t)
    presencia = {0: claves_vivienda_por_trimestre[(anio_t, trimestre_t)]}
    for offset in tr.OFFSETS_VISITA:
        destino = panel.sumar_trimestres(anio_t, trimestre_t, offset)
        tabla = claves_vivienda_por_trimestre.get(destino)
        if (tabla is None or es_historico(*destino) != historico_t
                or _cruza_ruptura_d13((anio_t, trimestre_t), destino)):
            presencia[offset] = None
        else:
            presencia[offset] = tabla
    return presencia


visitas_por_trimestre: dict[tuple[int, int], pd.DataFrame] = {}
for anio_t, trimestre_t in TODOS_LOS_TRIMESTRES:
    if (anio_t, trimestre_t) not in claves_vivienda_por_trimestre:
        continue
    visitas_por_trimestre[(anio_t, trimestre_t)] = tr.inferir_visita(_presencia_offsets(anio_t, trimestre_t))

todas_visitas = pd.concat(
    [df.assign(ANO4=a, TRIMESTRE=t) for (a, t), df in visitas_por_trimestre.items()], ignore_index=True)
print(f"{len(todas_visitas):,} viviendas-trimestre clasificadas")
print(todas_visitas["visita_inferida"].value_counts(dropna=False).sort_index())
print(f"ambigua: {todas_visitas['ambigua'].mean():.1%}")
print("n_determinables -- describe:")
print(todas_visitas["n_determinables"].describe())

### Antes/después del fix, distribución de visita 2012-2015

Recalculo la distribución de visita 2012-2015 con la regla VIEJA (solo publicación + cruce histórico/regular, sin la condición de ruptura D13) y la comparo contra la nueva (`todas_visitas`, ya con el fix aplicado) para esos mismos trimestres.

In [ ]:
def _presencia_offsets_sin_fix_d13(anio_t: int, trimestre_t: int) -> dict[int, pd.DataFrame | None]:
    historico_t = es_historico(anio_t, trimestre_t)
    presencia = {0: claves_vivienda_por_trimestre[(anio_t, trimestre_t)]}
    for offset in tr.OFFSETS_VISITA:
        destino = panel.sumar_trimestres(anio_t, trimestre_t, offset)
        tabla = claves_vivienda_por_trimestre.get(destino)
        presencia[offset] = None if (tabla is None or es_historico(*destino) != historico_t) else tabla
    return presencia


_ventana_antes_despues = [(a, t) for a in (2012, 2013, 2014, 2015) for t in (1, 2, 3, 4)
                           if (a, t) in claves_vivienda_por_trimestre]
_visitas_viejo = {q: tr.inferir_visita(_presencia_offsets_sin_fix_d13(*q)) for q in _ventana_antes_despues}
_antes = pd.concat([df.assign(ANO4=a, TRIMESTRE=t) for (a, t), df in _visitas_viejo.items()], ignore_index=True)
_despues = todas_visitas[todas_visitas.set_index(["ANO4", "TRIMESTRE"]).index.isin(_ventana_antes_despues)]

_dist_antes = _antes.groupby(["ANO4", "TRIMESTRE"])["visita_inferida"].value_counts(
    normalize=True, dropna=False).unstack().round(3)
_dist_despues = _despues.groupby(["ANO4", "TRIMESTRE"])["visita_inferida"].value_counts(
    normalize=True, dropna=False).unstack().round(3)
print("--- ANTES del fix (regla vieja: solo publicación + cruce histórico/regular) ---")
display(_dist_antes)
print("--- DESPUÉS del fix (con la condición de ruptura D13) ---")
display(_dist_despues)

### Por qué quedan sin clasificar

`n_determinables` más bajo cerca de los bordes de la serie, de los huecos de publicación
(2007T3, 2015T3-2016T1) y del cruce de era (2016T2-2016T3, D7) -- consecuencia natural del
método, no un caso especial en el código (ver D12). Tabla de `n_determinables` promedio y tasa de
`ambigua` por trimestre, para ubicar visualmente dónde se concentra.

In [ ]:
resumen_determinables = todas_visitas.groupby(["ANO4", "TRIMESTRE"]).agg(
    n_determinables_prom=("n_determinables", "mean"), tasa_ambigua=("ambigua", "mean"),
    n_viviendas=("CODUSU", "size"),
).reset_index()
peor = resumen_determinables.nsmallest(10, "n_determinables_prom")
print("10 trimestres con menor n_determinables promedio (bordes de serie / huecos / cruce de era):")
display(peor)

### Eslabones del panel: pares_h1 / pares_h4 por trimestre de origen

In [ ]:
COLUMNAS_PARES = ["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th", "AGLOMERADO", "identidad_confirmada"]


def _leer_pares(base_dir: Path, anio: int, trimestre: int) -> pd.DataFrame | None:
    p = base_dir / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    if not p.exists():
        return None
    return pd.read_parquet(p, columns=COLUMNAS_PARES)


pares_h1_por_origen: dict[tuple[int, int], pd.DataFrame] = {}
pares_h4_por_origen: dict[tuple[int, int], pd.DataFrame] = {}
for anio, trimestre in TODOS_LOS_TRIMESTRES:
    p1 = _leer_pares(PANEL / "pares_h1", anio, trimestre)
    if p1 is not None:
        pares_h1_por_origen[(anio, trimestre)] = p1
    p4 = _leer_pares(PANEL / "pares_h4", anio, trimestre)
    if p4 is not None:
        pares_h4_por_origen[(anio, trimestre)] = p4

print(f"{len(pares_h1_por_origen)} orígenes con pares_h1, {len(pares_h4_por_origen)} con pares_h4")

### Encadenamiento: t→t+1, t→t+4, t+4→t+5, verificado contra t+1→t+5

Reusa los pares ya construidos por el notebook 02 (no reconstruye desde `02_nucleo/`).

In [ ]:
def ruta_trayectoria(base_dir: Path, anio: int, trimestre: int) -> Path:
    return base_dir / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"


trayectorias_por_origen: dict[tuple[int, int], pd.DataFrame] = {}
n_escritos = n_reusados = n_sin_eslabones = 0
for anio_t0, trimestre_t0 in trimestres_origen:
    t0_1 = panel.sumar_trimestres(anio_t0, trimestre_t0, 1)
    t0_4 = panel.sumar_trimestres(anio_t0, trimestre_t0, 4)

    if ((anio_t0, trimestre_t0) not in pares_h1_por_origen or (anio_t0, trimestre_t0) not in pares_h4_por_origen
            or t0_4 not in pares_h1_por_origen):
        n_sin_eslabones += 1
        continue

    destino = ruta_trayectoria(TRAYECTORIAS / "trayectorias", anio_t0, trimestre_t0)
    origenes_parquet = [
        PANEL / "pares_h1" / f"ANO4={anio_t0}" / f"TRIMESTRE={trimestre_t0}" / "data.parquet",
        PANEL / "pares_h4" / f"ANO4={anio_t0}" / f"TRIMESTRE={trimestre_t0}" / "data.parquet",
        PANEL / "pares_h1" / f"ANO4={t0_4[0]}" / f"TRIMESTRE={t0_4[1]}" / "data.parquet",
    ]
    desactualizada = REHACER or not destino.exists() or any(pm.particion_desactualizada(o, destino) for o in origenes_parquet)

    if not desactualizada:
        trayectoria = pd.read_parquet(destino)
        n_reusados += 1
    else:
        encadenada = tr.encadenar_trayectoria(
            pares_h1_por_origen[(anio_t0, trimestre_t0)], pares_h4_por_origen[(anio_t0, trimestre_t0)],
            pares_h1_por_origen[t0_4])
        pares_h4_t1 = pares_h4_por_origen.get(t0_1)
        if pares_h4_t1 is not None:
            trayectoria = tr.verificar_contra_h4_t1(encadenada, pares_h4_t1)
        else:
            trayectoria = encadenada.assign(
                identidad_confirmada_verificacion=pd.array([pd.NA] * len(encadenada), dtype="boolean"),
                presente_en_verificacion=False)
        trayectoria["ANO4_t"] = anio_t0
        trayectoria["TRIMESTRE_t"] = trimestre_t0
        destino.parent.mkdir(parents=True, exist_ok=True)
        trayectoria.to_parquet(destino, index=False)
        n_escritos += 1

    trayectorias_por_origen[(anio_t0, trimestre_t0)] = trayectoria

todas_trayectorias = (pd.concat(trayectorias_por_origen.values(), ignore_index=True)
                       if trayectorias_por_origen else pd.DataFrame())
print(f"{len(trayectorias_por_origen)} orígenes con trayectoria ({n_escritos} escritos, {n_reusados} reusados, "
      f"{n_sin_eslabones} sin los 3 eslabones -- no completable, ver Sección 2)")
if not todas_trayectorias.empty:
    n_vinculada = len(todas_trayectorias)
    n_identidad = int((todas_trayectorias["identidad_confirmada_trayectoria"] == True).sum())
    n_sin_verificacion = int((~todas_trayectorias["presente_en_verificacion"]).sum())
    print(f"vinculada (clave exacta en los 3 eslabones): {n_vinculada:,}")
    print(f"identidad_confirmada_trayectoria: {n_identidad:,} ({n_identidad / n_vinculada:.1%})")
    print(f"sin presencia en la verificación cruzada pares_h4(t+1): {n_sin_verificacion:,} "
          f"({n_sin_verificacion / n_vinculada:.1%}) -- anomalía si > 0, no se descarta")

### Chequeo 3a — ¿las trayectorias vienen de viviendas en visita 1?

`encadenar_trayectoria` no filtra por visita -- solo exige la clave exacta en los 3 eslabones.
Si el método de inferencia de visita y la construcción de la cohorte son consistentes, casi toda
trayectoria debería originarse en una vivienda `visita_inferida == 1` en `t0` (así se define
"cohorte de entrada" en las Secciones 2 y 4). Se verifica cruzando contra `todas_visitas`, no se
asume.

In [ ]:
chequeo_visita1 = todas_trayectorias[["CODUSU", "AGLOMERADO", "ANO4_t", "TRIMESTRE_t"]].merge(
    todas_visitas[["CODUSU", "AGLOMERADO", "ANO4", "TRIMESTRE", "visita_inferida"]],
    left_on=["CODUSU", "AGLOMERADO", "ANO4_t", "TRIMESTRE_t"],
    right_on=["CODUSU", "AGLOMERADO", "ANO4", "TRIMESTRE"], how="left",
)
no_visita1 = (chequeo_visita1["visita_inferida"] != 1) | chequeo_visita1["visita_inferida"].isna()
print(f"trayectorias cuya vivienda de origen NO es visita_inferida==1 en t0: "
      f"{int(no_visita1.sum()):,} de {len(chequeo_visita1):,} ({no_visita1.mean():.2%})")
if no_visita1.any():
    print("distribución de visita_inferida real en los casos que no son visita 1:")
    display(chequeo_visita1.loc[no_visita1, "visita_inferida"].value_counts(dropna=False))

## Sección 2 — conteos y cortes

`cohorte_completable` para cada origen candidato: existencia teórica de `t0, t0+1, t0+4, t0+5`
contra `trimestres_publicados()`, y cruce de era en el eslabón `h=4` (D7). Se cruza contra los
conteos observados (cuántas viviendas en visita 1 en `t0` terminan vinculadas / con identidad
confirmada).

**Nota D1**: en los 6 trimestres posteriores a la ampliación de aglomerados de 2006T3
(2006T4-2008T1), parte de la cohorte de visita 1 corresponde a aglomerados nuevos sin historia
previa -- queda señalado en la tabla (columna `posterior_ampliacion_2006t3`), no se excluye.

**Nota D13**: `2013T1, 2013T2, 2013T3` son teóricamente completables (`t0,t0+1,t0+4,t0+5` están
publicados, misma era) pero tienen una ruptura real de `CODUSU` en el eslabón `h=4` saliente (ver
diagnóstico al final del notebook) -- quedan marcados aparte (`corte_estructural_d13`) y la tasa
de completitud se reporta con y sin ellos, para no medir esa ruptura como si fuera atrición real.

In [ ]:
PUBLICADOS_SET = set(TODOS_LOS_TRIMESTRES)
AMPLIACION_2006T3_HASTA = panel.sumar_trimestres(2006, 3, 5)  # 6 trimestres incluyendo el propio 2006T3
CORTE_ESTRUCTURAL_D13 = {(2013, 1), (2013, 2), (2013, 3)}  # ruptura de CODUSU en el eslabón h=4 saliente, D13

filas_cohorte = []
for anio_t0, trimestre_t0 in trimestres_origen:
    chequeo = tr.cohorte_completable(anio_t0, trimestre_t0, PUBLICADOS_SET, es_historico)
    visitas_t0 = visitas_por_trimestre.get((anio_t0, trimestre_t0))
    if visitas_t0 is None:
        continue
    cohorte_v1 = visitas_t0[visitas_t0["visita_inferida"] == 1][["CODUSU", "AGLOMERADO"]]
    n_v1 = len(cohorte_v1)

    trayectoria_t0 = trayectorias_por_origen.get((anio_t0, trimestre_t0))
    if trayectoria_t0 is not None and n_v1 > 0:
        vinculados_v1 = trayectoria_t0.merge(cohorte_v1, on=["CODUSU", "AGLOMERADO"], how="inner")
        n_vinculada = len(vinculados_v1)
        n_identidad = int((vinculados_v1["identidad_confirmada_trayectoria"] == True).sum())
    else:
        n_vinculada = n_identidad = 0

    filas_cohorte.append({
        "ANO4_t0": anio_t0, "TRIMESTRE_t0": trimestre_t0, "era": "historico" if es_historico(anio_t0, trimestre_t0) else "regular",
        "completable": chequeo["completable"], "motivo": chequeo["motivo"],
        "posterior_ampliacion_2006t3": (anio_t0, trimestre_t0) >= (2006, 3) and (anio_t0, trimestre_t0) <= AMPLIACION_2006T3_HASTA,
        "corte_estructural_d13": (anio_t0, trimestre_t0) in CORTE_ESTRUCTURAL_D13,
        "n_vivienda_visita1": n_v1, "n_vinculada": n_vinculada, "n_identidad_confirmada": n_identidad,
    })

completitud_cohorte = pd.DataFrame(filas_cohorte)
completitud_cohorte.to_csv(META / "trayectorias_completitud_cohorte.csv", index=False)
print(f"{len(completitud_cohorte)} cohortes evaluadas, {completitud_cohorte['completable'].sum()} completables en teoría "
      f"({completitud_cohorte['corte_estructural_d13'].sum()} de ellas con corte estructural D13)")
print(completitud_cohorte.loc[~completitud_cohorte["completable"], "motivo"].value_counts())

completables = completitud_cohorte[completitud_cohorte["completable"] & (completitud_cohorte["n_vivienda_visita1"] > 0)]
completables_sin_d13 = completables[~completables["corte_estructural_d13"]]
for etiqueta, df_tasa in (("todas las completables", completables), ("excluyendo corte estructural D13", completables_sin_d13)):
    if len(df_tasa):
        num_vinc, den = int(df_tasa["n_vinculada"].sum()), int(df_tasa["n_vivienda_visita1"].sum())
        num_id = int(df_tasa["n_identidad_confirmada"].sum())
        print(f"tasa de completitud observada ({etiqueta}, n_origenes={len(df_tasa)}): "
              f"vinculada={num_vinc / den:.1%} ({num_vinc:,}/{den:,}), "
              f"identidad_confirmada={num_id / den:.1%} ({num_id:,}/{den:,})")

## Sección 3a — distribución de visita por trimestre (sin ponderar)

Se espera ~25% por grupo (1-4) en estado estacionario.

In [ ]:
distribucion_visita = (
    todas_visitas.groupby(["ANO4", "TRIMESTRE"])["visita_inferida"]
    .value_counts(normalize=True, dropna=False).unstack().sort_index()
)
distribucion_visita.columns = [f"visita_{int(c)}" if pd.notna(c) else "no_clasificable" for c in distribucion_visita.columns]

fig, ax = plt.subplots(figsize=(16, 5))
eje_x = [f"{a}T{t}" for a, t in distribucion_visita.index]
for col in [c for c in distribucion_visita.columns if c.startswith("visita_")]:
    ax.plot(eje_x, distribucion_visita[col], label=col, linewidth=1)
ax.axhline(0.25, color="gray", linestyle="--", linewidth=1, label="25% de referencia")
ax.set_xticks(eje_x[::8])
ax.set_xticklabels(eje_x[::8], rotation=90)
ax.set_ylabel("% de viviendas")
ax.set_title("Distribución de visita inferida por trimestre")
ax.legend()
plt.tight_layout()
plt.show()

## Sección 3b — efecto de visita (panel conditioning)

Dentro de cada trimestre, por número de visita inferida: tasa de `ingreso_no_declarado` (solo
era regular, D3), `hay_ocupado` del hogar, y distribución de `grupo_ingreso` (quintil vía
`DECCFR`). Puramente descriptivo, sin modelo ni causalidad.

In [ ]:
filas_overlapping = []
for anio, trimestre in trimestres_origen:
    visitas_t = visitas_por_trimestre.get((anio, trimestre))
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO)
    if visitas_t is None or hogar is None or individual is None:
        continue
    hogar = hogar.copy()
    hogar["grupo_ingreso"] = calcular_grupo_ingreso(hogar)
    hogar["hay_ocupado"] = hogar.set_index(["CODUSU", "NRO_HOGAR"]).index.isin(
        individual.loc[individual["ESTADO"] == 1, ["CODUSU", "NRO_HOGAR"]].set_index(["CODUSU", "NRO_HOGAR"]).index)
    hogar = hogar.merge(visitas_t[["CODUSU", "AGLOMERADO", "visita_inferida"]], on=["CODUSU", "AGLOMERADO"], how="inner")

    for visita, g in hogar.groupby("visita_inferida", dropna=True):
        fila = {
            "ANO4": anio, "TRIMESTRE": trimestre, "era": "historico" if es_historico(anio, trimestre) else "regular",
            "visita_inferida": int(visita), "n_hogares": len(g), "tasa_hay_ocupado": g["hay_ocupado"].mean(),
        }
        if not es_historico(anio, trimestre):
            fila["tasa_ingreso_no_declarado"] = g["ingreso_no_declarado"].mean()
        for q in ORDEN_GRUPOS_INGRESO:
            fila[f"pct_{q}"] = (g["grupo_ingreso"] == q).mean()
        filas_overlapping.append(fila)

overlapping_visita = pd.DataFrame(filas_overlapping)
overlapping_visita.to_csv(META / "trayectorias_overlapping_visita.csv", index=False)
print(f"{len(overlapping_visita)} filas trimestre x visita")
display(overlapping_visita.groupby("visita_inferida")[["tasa_hay_ocupado"] + [f"pct_{q}" for q in ORDEN_QUINTILES]].mean().round(3))
if "tasa_ingreso_no_declarado" in overlapping_visita.columns:
    display(overlapping_visita.groupby("visita_inferida")["tasa_ingreso_no_declarado"].mean().round(3))

## Sección 4 — selectividad de la atrición

Universo: hogares cuya **vivienda** tiene `visita_inferida == 1` (no `ambigua`) en `t0`, solo
para orígenes `t0` **completables** y **sin corte estructural D13** (2013T1-T3 quedan afuera --
ver Sección 2, si no la ruptura de `CODUSU` se mide como atrición real de los hogares). La
pérdida se clasifica por el **patrón real de presencia** (`tr.clasificar_patron_apariciones` +
`tr.clasificar_perdida_por_patron`), no por el último eslabón alcanzado -- un hueco con
reaparición (p. ej. `"1011"`) se reporta aparte, no se confunde con "se pierde en t0+5". El
bucket `completa` se separa en `completa_identidad_confirmada` e `completa_vinculada_sin_identidad`.
Mismas categorías del notebook 03. Cada tabla sale 3 veces: sin ponderar, `PONDERA` (todas las
categorías, ambas eras) y `PONDIH` (solo ingreso-declarado, era regular -- D3).

In [ ]:
origenes_validos_sel = set(
    completitud_cohorte.loc[
        completitud_cohorte["completable"],
        ["ANO4_t0", "TRIMESTRE_t0"],
    ].itertuples(index=False, name=None)
) - VISITA_AFECTADA_D13


def _tabla_categorica(df: pd.DataFrame, columna: str, pesos: str | None) -> pd.Series:
    if pesos is None:
        return df.groupby("estado")[columna].value_counts(normalize=True).unstack()
    return df.groupby("estado").apply(
        lambda g: g.groupby(columna)[pesos].sum() / g[pesos].sum() if g[pesos].sum() else np.nan
    )


filas_selectividad = []
n_saltados_d13 = 0
for anio_t0, trimestre_t0 in trimestres_origen:
    if (anio_t0, trimestre_t0) not in origenes_validos_sel:
        if (anio_t0, trimestre_t0) in VISITA_AFECTADA_D13:
            n_saltados_d13 += 1
        continue

    visitas_t0 = visitas_por_trimestre.get((anio_t0, trimestre_t0))
    if visitas_t0 is None:
        continue
    cohorte_v1 = visitas_t0.loc[visitas_t0["visita_inferida"] == 1, ["CODUSU", "AGLOMERADO"]]
    if cohorte_v1.empty:
        continue

    hogar_t0 = pm.cargar_hogar(anio_t0, trimestre_t0, COLUMNAS_HOGAR, NUCLEO)
    individual_t0 = pm.cargar_individual(anio_t0, trimestre_t0, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO)
    if hogar_t0 is None or individual_t0 is None:
        continue
    hogar_t0 = hogar_t0.merge(cohorte_v1, on=["CODUSU", "AGLOMERADO"], how="inner")
    if hogar_t0.empty:
        continue
    historico = es_historico(anio_t0, trimestre_t0)

    t0_1 = panel.sumar_trimestres(anio_t0, trimestre_t0, 1)
    t0_4 = panel.sumar_trimestres(anio_t0, trimestre_t0, 4)
    t0_5 = panel.sumar_trimestres(anio_t0, trimestre_t0, 5)
    hogares_cohorte = tr.claves_hogar(hogar_t0)
    patron = tr.clasificar_patron_apariciones(
        hogares_cohorte, claves_hogar_por_trimestre.get(t0_1), claves_hogar_por_trimestre.get(t0_4),
        claves_hogar_por_trimestre.get(t0_5))
    patron["estado"] = tr.clasificar_perdida_por_patron(patron["patron"])

    trayectoria_t0 = trayectorias_por_origen.get((anio_t0, trimestre_t0))
    if trayectoria_t0 is not None:
        ident = trayectoria_t0[["CODUSU", "NRO_HOGAR", "AGLOMERADO", "identidad_confirmada_trayectoria"]]
        patron = patron.merge(ident, on=["CODUSU", "NRO_HOGAR", "AGLOMERADO"], how="left")
        es_completa = patron["estado"] == "completa"
        patron.loc[es_completa & (patron["identidad_confirmada_trayectoria"] == True), "estado"] = "completa_identidad_confirmada"
        patron.loc[es_completa & (patron["identidad_confirmada_trayectoria"] != True), "estado"] = "completa_vinculada_sin_identidad"

    info_jefe = calcular_info_jefe(individual_t0)
    situacion = calcular_situacion_laboral(individual_t0)
    n_miembros = individual_t0.groupby(["CODUSU", "NRO_HOGAR"]).size()
    ancla = calcular_componentes_ancla(hogar_t0.set_index(["CODUSU", "NRO_HOGAR"]), n_miembros).reset_index()

    base = hogar_t0.copy()
    base["grupo_ingreso"] = calcular_grupo_ingreso(base)
    base = base.merge(info_jefe, on=["CODUSU", "NRO_HOGAR"], how="left")
    base["nivel_ed_bucket_jefe"] = bucket_nivel_ed(base["nivel_ed_jefe"])
    base = base.merge(situacion, on=["CODUSU", "NRO_HOGAR"], how="left")
    base = base.merge(ancla, on=["CODUSU", "NRO_HOGAR"], how="left")
    base = base.merge(n_miembros.rename("n_miembros").reset_index(), on=["CODUSU", "NRO_HOGAR"], how="left")
    base["tamano"] = base["n_miembros"].clip(upper=6).astype("Int64").astype(str)
    base.loc[base["n_miembros"] >= 6, "tamano"] = "6+"
    base = base.merge(patron[["CODUSU", "NRO_HOGAR", "AGLOMERADO", "patron", "estado"]],
                       on=["CODUSU", "NRO_HOGAR", "AGLOMERADO"], how="left")

    base["era"] = "historico" if historico else "regular"
    base["ANO4_t0"], base["TRIMESTRE_t0"] = anio_t0, trimestre_t0
    filas_selectividad.append(base)

selectividad_base = pd.concat(filas_selectividad, ignore_index=True) if filas_selectividad else pd.DataFrame()
print(f"{len(selectividad_base):,} hogares (vivienda visita 1, cohortes completables fuera de VISITA_AFECTADA_D13) "
      f"acumulados para selectividad -- {n_saltados_d13} orígenes saltados por D13 (de {len(VISITA_AFECTADA_D13)} afectados: "
      f"{sorted(VISITA_AFECTADA_D13)})")
if len(selectividad_base):
    print(selectividad_base["estado"].value_counts(dropna=False))

In [ ]:
CATEGORICAS = ["grupo_ingreso", "situacion_laboral", "ancla_con_deficit", "nivel_ed_bucket_jefe", "tamano", "sexo_jefe"]
tablas_sel = []
for era, df_era in selectividad_base.groupby("era"):
    for ponderacion, col_peso in (("sin_ponderar", None), ("PONDERA", "pondera_jefe")):
        for columna in CATEGORICAS:
            tabla = _tabla_categorica(df_era, columna, col_peso).reset_index().melt(
                id_vars="estado", var_name=columna, value_name="proporcion")
            tabla["era"], tabla["ponderacion"], tabla["categoria"] = era, ponderacion, columna
            tabla = tabla.rename(columns={columna: "valor"})
            tablas_sel.append(tabla)

regular = selectividad_base[selectividad_base["era"] == "regular"].copy()
if len(regular):
    regular["ingreso_declarado_cat"] = (~regular["ingreso_no_declarado"].astype("boolean")).map({True: "declarado", False: "no_declarado"})
    for ponderacion, col_peso in (("sin_ponderar", None), ("PONDERA", "pondera_jefe"), ("PONDIH", "PONDIH")):
        tabla = _tabla_categorica(regular, "ingreso_declarado_cat", col_peso).reset_index().melt(
            id_vars="estado", var_name="valor", value_name="proporcion")
        tabla["era"], tabla["ponderacion"], tabla["categoria"] = "regular", ponderacion, "ingreso_declarado"
        tablas_sel.append(tabla)

historico = selectividad_base[selectividad_base["era"] == "historico"].copy()
if len(historico):
    historico["ipcf_presente"] = historico["IPCF"].notna().map({True: "presente", False: "ausente"})
    for ponderacion, col_peso in (("sin_ponderar", None), ("PONDERA", "pondera_jefe")):
        tabla = _tabla_categorica(historico, "ipcf_presente", col_peso).reset_index().melt(
            id_vars="estado", var_name="valor", value_name="proporcion")
        tabla["era"], tabla["ponderacion"], tabla["categoria"] = "historico", ponderacion, "ipcf_presente"
        tablas_sel.append(tabla)

atricion_selectiva = pd.concat(tablas_sel, ignore_index=True) if tablas_sel else pd.DataFrame()
atricion_selectiva.to_csv(META / "trayectorias_atricion_selectiva.csv", index=False)
print(f"{len(atricion_selectiva):,} filas en trayectorias_atricion_selectiva.csv")
if len(atricion_selectiva):
    sub = atricion_selectiva[(atricion_selectiva["categoria"] == "grupo_ingreso") & (atricion_selectiva["ponderacion"] == "sin_ponderar")
                              & (atricion_selectiva["era"] == "regular")]
    if len(sub):
        display(sub.pivot(index="estado", columns="valor", values="proporcion").round(3))

## Sección 5 — N para diseños dinámicos

(a) `pares_h1(t)` ∩ `pares_h4(t+1)` -- cambio t→t+1 predice t+1→t+5 (h=4), no requiere el eslabón
t→t+4; (b) `pares_h4(t)` ∩ `pares_h1(t+4)` -- cambio t→t+4 predice t+4→t+5 (h=1), no requiere
t→t+1; (c) trayectoria completa de 4 visitas (los 3 eslabones). Cada diseño se cuenta en 2
variantes: `vinculada` (clave exacta) e `identidad_confirmada` (ambos eslabones con identidad
confirmada). Umbrales de N **[inferido]** -- no son criterio de INDEC (que usa coeficiente de
variación, no un piso fijo). **Una vez exista una etiqueta de deterioro, el umbral relevante no
va a ser el N total por trimestre sino el N de la clase minoritaria** -- con estos umbrales
genéricos solo se estima una cota superior optimista.

In [ ]:
filas_a, filas_b = [], []
n_a_saltados = n_b_saltados = 0
for anio_t0, trimestre_t0 in trimestres_origen:
    t0_1 = panel.sumar_trimestres(anio_t0, trimestre_t0, 1)
    t0_4 = panel.sumar_trimestres(anio_t0, trimestre_t0, 4)

    if (anio_t0, trimestre_t0) in pares_h1_por_origen and t0_1 in pares_h4_por_origen:
        tabla_a = tr.encadenar_dos_eslabones(pares_h1_por_origen[(anio_t0, trimestre_t0)], pares_h4_por_origen[t0_1])
        tabla_a["ANO4_t"], tabla_a["TRIMESTRE_t"] = anio_t0, trimestre_t0
        filas_a.append(tabla_a)
    else:
        n_a_saltados += 1

    if (anio_t0, trimestre_t0) in pares_h4_por_origen and t0_4 in pares_h1_por_origen:
        tabla_b = tr.encadenar_dos_eslabones(pares_h4_por_origen[(anio_t0, trimestre_t0)], pares_h1_por_origen[t0_4])
        tabla_b["ANO4_t"], tabla_b["TRIMESTRE_t"] = anio_t0, trimestre_t0
        filas_b.append(tabla_b)
    else:
        n_b_saltados += 1

disenos_a = pd.concat(filas_a, ignore_index=True) if filas_a else pd.DataFrame()
disenos_b = pd.concat(filas_b, ignore_index=True) if filas_b else pd.DataFrame()

n_a = tr.contar_n_diseno(disenos_a, "identidad_confirmada_ambos").assign(diseno="a") if not disenos_a.empty else pd.DataFrame()
n_b = tr.contar_n_diseno(disenos_b, "identidad_confirmada_ambos").assign(diseno="b") if not disenos_b.empty else pd.DataFrame()
n_c = tr.contar_n_diseno(todas_trayectorias, "identidad_confirmada_trayectoria").assign(diseno="c") if not todas_trayectorias.empty else pd.DataFrame()

n_disenos = pd.concat([n_a, n_b, n_c], ignore_index=True)
if not n_disenos.empty:
    n_disenos["era"] = n_disenos.apply(lambda f: "historico" if es_historico(f["ANO4_t"], f["TRIMESTRE_t"]) else "regular", axis=1)
n_disenos.to_csv(META / "trayectorias_n_disenos.csv", index=False)

print(f"orígenes sin datos -- diseño a: {n_a_saltados}, diseño b: {n_b_saltados} (eslabón faltante en algún extremo)")
print(f"N total (vinculada) por diseño: a={n_a['vinculada'].sum() if len(n_a) else 0:,}, "
      f"b={n_b['vinculada'].sum() if len(n_b) else 0:,}, c={n_c['vinculada'].sum() if len(n_c) else 0:,}")
if not n_disenos.empty:
    display(n_disenos.groupby(["diseno", "era"])[["vinculada", "identidad_confirmada"]].sum())

for umbral in (30, 100, 500):
    for etiqueta, tabla_d in (("a", n_a), ("b", n_b), ("c", n_c)):
        if tabla_d.empty:
            continue
        viables_vinc = int((tabla_d["vinculada"] >= umbral).sum())
        viables_id = int((tabla_d["identidad_confirmada"] >= umbral).sum())
        print(f"diseño {etiqueta}, umbral N>={umbral}: {viables_vinc} trimestres viables (vinculada), "
              f"{viables_id} (identidad_confirmada), de {len(tabla_d)} orígenes con datos")

### Chequeo 3b — qué orígenes completables quedan bajo umbral, y por qué

Sobre los orígenes teóricamente completables (Sección 2), para cada diseño: cuáles tienen
`N < 500` (el umbral más estricto de los 3) y el motivo -- distinguiendo explícitamente el corte
estructural D13 (2013-2014) de cualquier otro caso. El eslabón `h=4` roto afecta a orígenes
distintos según el diseño: (a) usa `pares_h4(t+1)`, afectado en `t0 ∈ {2012T4, 2013T1, 2013T2}`;
(b)/(c) usan `pares_h4(t)` directo, afectados en `t0 ∈ {2013T1, 2013T2, 2013T3}`.

In [ ]:
AFECTADOS_D13_POR_DISENO = {
    "a": {(2012, 4), (2013, 1), (2013, 2)},
    "b": CORTE_ESTRUCTURAL_D13,
    "c": CORTE_ESTRUCTURAL_D13,
}
completables_set = set(completitud_cohorte.loc[completitud_cohorte["completable"], ["ANO4_t0", "TRIMESTRE_t0"]]
                        .itertuples(index=False, name=None))

for etiqueta, tabla_d in (("a", n_a), ("b", n_b), ("c", n_c)):
    if tabla_d.empty:
        continue
    indexada = tabla_d.set_index(["ANO4_t", "TRIMESTRE_t"])
    filas_bajo_umbral = []
    for origen in sorted(completables_set):
        es_d13 = origen in AFECTADOS_D13_POR_DISENO[etiqueta]
        if origen not in indexada.index:
            motivo = "corte estructural D13 -- sin datos (0 filas)" if es_d13 else "sin datos para este diseño (eslabón faltante)"
            filas_bajo_umbral.append({"ANO4_t": origen[0], "TRIMESTRE_t": origen[1], "n_vinculada": 0, "motivo": motivo})
            continue
        n_vinc = int(indexada.loc[origen, "vinculada"])
        if n_vinc >= 500:
            continue
        motivo = "corte estructural D13 (ruptura CODUSU 2013-2014)" if es_d13 else "otro (revisar)"
        filas_bajo_umbral.append({"ANO4_t": origen[0], "TRIMESTRE_t": origen[1], "n_vinculada": n_vinc, "motivo": motivo})

    if filas_bajo_umbral:
        print(f"--- diseño {etiqueta}: orígenes completables con N < 500 ---")
        display(pd.DataFrame(filas_bajo_umbral))
    else:
        print(f"diseño {etiqueta}: todos los orígenes completables superan N >= 500")

## Sección 6 — ilustración exploratoria: quintil en dos pasos

Sobre `identidad_confirmada_trayectoria == True`: compara la transición directa `t0+1 → t0+5`
("un paso", ya disponible vía el eslabón de verificación) contra la distribución de quintil en
`t0+5` condicional a quintil en `t0+1` **y** en `t0` ("dos pasos") -- ¿agrega información el
quintil de `t0`? Se muestra el `n` crudo de cada celda junto a la proporción, y se marcan las
celdas con `n < 100` (lectura menos confiable). Puramente ilustrativo, no es la etiqueta, no se
interpreta como causal.

In [ ]:
def _quintil_en(anio: int, trimestre: int, claves: pd.DataFrame) -> pd.Series:
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR, NUCLEO)
    if hogar is None:
        return pd.Series(dtype="object")
    hogar = hogar.copy()
    hogar["grupo_ingreso"] = calcular_grupo_ingreso(hogar)
    return claves.merge(hogar[["CODUSU", "NRO_HOGAR", "grupo_ingreso"]], on=["CODUSU", "NRO_HOGAR"], how="left")["grupo_ingreso"]


filas_dos_pasos = []
for anio_t0, trimestre_t0 in trimestres_origen:
    trayectoria = trayectorias_por_origen.get((anio_t0, trimestre_t0))
    if trayectoria is None:
        continue
    completos = trayectoria[trayectoria["identidad_confirmada_trayectoria"] == True][["CODUSU", "NRO_HOGAR", "AGLOMERADO"]]
    if completos.empty:
        continue
    t0_1 = panel.sumar_trimestres(anio_t0, trimestre_t0, 1)
    t0_5 = panel.sumar_trimestres(anio_t0, trimestre_t0, 5)

    quintil_t0 = _quintil_en(anio_t0, trimestre_t0, completos)
    quintil_t0_1 = _quintil_en(*t0_1, completos)
    quintil_t0_5 = _quintil_en(*t0_5, completos)
    tabla = pd.DataFrame({"q_t0": quintil_t0.values, "q_t0_1": quintil_t0_1.values, "q_t0_5": quintil_t0_5.values})
    tabla = tabla[tabla["q_t0"].isin(ORDEN_QUINTILES) & tabla["q_t0_1"].isin(ORDEN_QUINTILES) & tabla["q_t0_5"].isin(ORDEN_QUINTILES)]
    filas_dos_pasos.append(tabla)

dos_pasos = pd.concat(filas_dos_pasos, ignore_index=True) if filas_dos_pasos else pd.DataFrame(columns=["q_t0", "q_t0_1", "q_t0_5"])
print(f"{len(dos_pasos):,} hogares con quintil observable en t0, t0+1 y t0+5 (identidad confirmada)")

if len(dos_pasos):
    n_un_paso = pd.crosstab(dos_pasos["q_t0_1"], dos_pasos["q_t0_5"]).reindex(index=ORDEN_QUINTILES, columns=ORDEN_QUINTILES, fill_value=0)
    un_paso = pd.crosstab(dos_pasos["q_t0_1"], dos_pasos["q_t0_5"], normalize="index").reindex(
        index=ORDEN_QUINTILES, columns=ORDEN_QUINTILES)
    print("--- Un paso: t0+1 -> t0+5 (proporción) ---")
    display(un_paso.round(3))
    print("--- Un paso: n por celda ---")
    display(n_un_paso)
    if (n_un_paso < 100).any().any():
        print("celdas con n < 100 (un paso):")
        print([(i, c) for i in n_un_paso.index for c in n_un_paso.columns if n_un_paso.loc[i, c] < 100])

    n_dos_pasos = dos_pasos.groupby(["q_t0_1", "q_t0"])["q_t0_5"].value_counts().unstack().reindex(columns=ORDEN_QUINTILES, fill_value=0)
    dos_pasos_tabla = dos_pasos.groupby(["q_t0_1", "q_t0"])["q_t0_5"].value_counts(normalize=True).unstack().reindex(
        columns=ORDEN_QUINTILES)
    print("--- Dos pasos: t0+5 | (t0+1, t0) (proporción) ---")
    display(dos_pasos_tabla.round(3))
    print("--- Dos pasos: n por celda ---")
    display(n_dos_pasos)
    celdas_bajo_100 = [(i, c) for i in n_dos_pasos.index for c in n_dos_pasos.columns if n_dos_pasos.loc[i, c] < 100]
    if celdas_bajo_100:
        print(f"{len(celdas_bajo_100)} celdas con n < 100 (dos pasos) -- lectura menos confiable ahí:")
        print(celdas_bajo_100)

    salida_dos_pasos = dos_pasos.groupby(["q_t0", "q_t0_1", "q_t0_5"]).size().rename("n").reset_index()
else:
    salida_dos_pasos = pd.DataFrame(columns=["q_t0", "q_t0_1", "q_t0_5", "n"])
salida_dos_pasos.to_csv(META / "trayectorias_transicion_dos_pasos.csv", index=False)

## Apéndice — diagnóstico exploratorio de la anomalía 2013-2014 (D13)

Solo para acotar la causa antes de redactar D13 -- no toca `panel.py` ni el notebook 02, no entra a `src/trayectorias.py`, no se persiste como CSV de `data/meta/`.

**Hallazgo de base (confirmado, matriz de la Sección 1)**: el eslabón `h=4` saliente de `2013T1/T2/T3` da 3,560 / 0 / 3,743 pares por `CODUSU` (vs. ~7,000-7,900 normal); `2013T4` es normal (7,622). La ruptura está en el propio `CODUSU`, no es un artefacto del join por `NRO_HOGAR`/`AGLOMERADO`.

**Diagnóstico 1 -- mecanismo por cohorte de entrada**: en cualquier trimestre conviven el grupo en 1.ª visita (vincula `h=1` y `h=4`) y el grupo en 2.ª visita (solo `h=4`, nunca `h=1` por construcción) -- de ahí que el solapamiento `h1∩h4` normal sea ~45% del `h4` (la porción que aporta el grupo de 1.ª visita, el único que estructuralmente pertenece a los dos). Construyo cada cohorte de entrada `E` (presentes en `E` y `E+1`, ausentes en `E-1`) entre 2012T3 y 2013T4, y mido su presencia en `E+4`/`E+5`.

In [ ]:
def _claves_vivienda_tupla(anio: int, trimestre: int) -> set[tuple]:
    return set(claves_vivienda_por_trimestre[(anio, trimestre)][["CODUSU", "AGLOMERADO"]].itertuples(index=False, name=None))


def _cohorte_entrada(anio_e: int, trimestre_e: int) -> set[tuple]:
    e_m1 = panel.sumar_trimestres(anio_e, trimestre_e, -1)
    e_p1 = panel.sumar_trimestres(anio_e, trimestre_e, 1)
    return (_claves_vivienda_tupla(anio_e, trimestre_e) & _claves_vivienda_tupla(*e_p1)) - _claves_vivienda_tupla(*e_m1)


_entradas_candidatas = [(2012, 3), (2012, 4), (2013, 1), (2013, 2), (2013, 3), (2013, 4)]
filas_mecanismo = []
for anio_e, trimestre_e in _entradas_candidatas:
    cohorte = _cohorte_entrada(anio_e, trimestre_e)
    e_p4 = panel.sumar_trimestres(anio_e, trimestre_e, 4)
    e_p5 = panel.sumar_trimestres(anio_e, trimestre_e, 5)
    n = len(cohorte)
    filas_mecanismo.append({
        "entrada": f"{anio_e}T{trimestre_e}", "n_cohorte": n,
        "pct_presente_Ep4": len(cohorte & _claves_vivienda_tupla(*e_p4)) / n if n else float("nan"),
        "pct_presente_Ep5": len(cohorte & _claves_vivienda_tupla(*e_p5)) / n if n else float("nan"),
    })
mecanismo_entrada = pd.DataFrame(filas_mecanismo)
display(mecanismo_entrada.style.format({"pct_presente_Ep4": "{:.1%}", "pct_presente_Ep5": "{:.1%}"}))

Confirma el mecanismo: exactamente dos cohortes de entrada rotas (2013T1, 2013T2 -- 0.0% limpio en ambas), el resto normal (43-47%). Explica las 4 filas de la tabla de `h=4` sin nada más: 2013T1 sobrevive solo con su grupo de 2.ª visita (entrada 2012T4, sano, nunca solapa `h1`); 2013T2 pierde los dos grupos que la alimentan (entrada 2013T1 y 2013T2, ambos rotos, de ahí los 0 pares); 2013T3 sobrevive solo con su grupo de 1.ª visita (entrada 2013T3, sano, sí solapa `h1`); 2013T4 tiene los dos grupos sanos (entrada 2013T3 y 2013T4).

**Diagnóstico 2 -- matching por composición, cohorte de entrada contra cohorte de entrada**: ¿las viviendas que entraron en 2013T1 (presentes en 2013T1 y 2013T2) tienen candidatas plausibles entre las que aparecen en 2014T1 con la misma estructura de entrada (presentes en 2014T1 y 2014T2, ausentes en todo 2013)? Comparado con la misma construcción en un año sin ruptura conocida (entrada 2011T1 → candidatas 2012T1) -- heurístico de bucket grueso (`AGLOMERADO + n_miembros + sexos`, edad del jefe/a dentro de la ventana `h=4` de D6), no un linkage definitivo.

In [ ]:
def _composicion(hogar: pd.DataFrame, individual: pd.DataFrame) -> pd.DataFrame:
    ind = individual[["CODUSU", "NRO_HOGAR", "CH03", "CH04", "CH06"]].copy()
    n_miembros = ind.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros")
    sexos = ind.groupby(["CODUSU", "NRO_HOGAR"])["CH04"].apply(lambda s: tuple(sorted(s.dropna().astype(int)))).rename("sexos")
    jefe = ind[ind["CH03"] == 1].drop_duplicates(["CODUSU", "NRO_HOGAR"]).set_index(["CODUSU", "NRO_HOGAR"])[["CH04", "CH06"]]
    jefe.columns = ["sexo_jefe", "edad_jefe"]
    base = hogar[["CODUSU", "NRO_HOGAR", "AGLOMERADO"]].set_index(["CODUSU", "NRO_HOGAR"])
    base = base.join(n_miembros).join(sexos).join(jefe)
    return base.reset_index()


def _candidatos_entrada(anio_c: int, trimestre_c: int, anios_excluir: list[int]) -> set[tuple]:
    cand = _claves_vivienda_tupla(anio_c, trimestre_c) & _claves_vivienda_tupla(*panel.sumar_trimestres(anio_c, trimestre_c, 1))
    ausentes = set()
    for anio in anios_excluir:
        for trimestre in (1, 2, 3, 4):
            if (anio, trimestre) in claves_vivienda_por_trimestre:
                ausentes |= _claves_vivienda_tupla(anio, trimestre)
    return cand - ausentes


def _tasa_matching_cohortes(anio_e: int, trimestre_e: int, anio_c: int, trimestre_c: int, anios_excluir: list[int]):
    entrada = _cohorte_entrada(anio_e, trimestre_e)
    candidatos_viv = _candidatos_entrada(anio_c, trimestre_c, anios_excluir)

    comp_e = _composicion(pm.cargar_hogar(anio_e, trimestre_e, COLUMNAS_HOGAR, NUCLEO), pm.cargar_individual(anio_e, trimestre_e, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO)).dropna(subset=["sexo_jefe", "edad_jefe"])
    comp_e = comp_e.merge(pd.DataFrame(list(entrada), columns=["CODUSU", "AGLOMERADO"]), on=["CODUSU", "AGLOMERADO"])
    comp_c = _composicion(pm.cargar_hogar(anio_c, trimestre_c, COLUMNAS_HOGAR, NUCLEO), pm.cargar_individual(anio_c, trimestre_c, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO)).dropna(subset=["sexo_jefe", "edad_jefe"])
    comp_c = comp_c.merge(pd.DataFrame(list(candidatos_viv), columns=["CODUSU", "AGLOMERADO"]), on=["CODUSU", "AGLOMERADO"])

    minimo, maximo = panel.TOLERANCIA_EDAD_POR_HORIZONTE[4]
    pares_candidatos = comp_e.merge(comp_c, on=["AGLOMERADO", "n_miembros", "sexos"], suffixes=("_e", "_c"))
    pares_candidatos = pares_candidatos[
        (pares_candidatos["sexo_jefe_e"] == pares_candidatos["sexo_jefe_c"])
        & (pares_candidatos["edad_jefe_c"] - pares_candidatos["edad_jefe_e"]).between(minimo, maximo)
    ]
    n_candidatos = pares_candidatos.groupby(["CODUSU_e", "NRO_HOGAR_e"]).size()
    n_candidatos = n_candidatos.reindex(comp_e.set_index(["CODUSU", "NRO_HOGAR"]).index, fill_value=0)
    tiene_match = n_candidatos > 0
    tasa = tiene_match.mean() if len(comp_e) else float("nan")
    return len(entrada), len(candidatos_viv), len(comp_e), int(tiene_match.sum()), tasa


n_e_2013, n_c_2014, n_eval_2013, n_match_2013, tasa_2013 = _tasa_matching_cohortes(2013, 1, 2014, 1, [2013])
n_e_2011, n_c_2012, n_eval_2011, n_match_2011, tasa_2011 = _tasa_matching_cohortes(2011, 1, 2012, 1, [2011])

print(f"Anomalía -- entrada 2013T1 (n={n_e_2013:,}) vs. candidatos 2014T1 (n={n_c_2014:,}): "
      f"{n_eval_2013:,} evaluables, {n_match_2013:,} con >=1 candidato -- tasa={tasa_2013:.1%}")
print(f"Base normal -- entrada 2011T1 (n={n_e_2011:,}) vs. candidatos 2012T1 (n={n_c_2012:,}): "
      f"{n_eval_2011:,} evaluables, {n_match_2011:,} con >=1 candidato -- tasa={tasa_2011:.1%}")

**Diagnóstico 3 -- ¿son los pares `h=4` sobrevivientes de 2013T1/2013T3 coincidencias de `CODUSU`?** Si lo fueran, `identidad_confirmada` (que exige sexo+edad del jefe/a dentro de la ventana D6) daría una tasa cercana al azar, no la tasa normal. Mido también cuánto solapan esos pares con los eslabones `h=1` en las dos puntas (`pares_h1(t0)`, usado por el diseño c, y `pares_h1(t0+4)`, usado por el diseño b), contra 2012T1 como base normal.

In [ ]:
def _fila_diagnostico_h4(anio_t: int, trimestre_t: int) -> dict:
    anio_th, trimestre_th = panel.sumar_trimestres(anio_t, trimestre_t, 4)
    hogar_t, hogar_th = pm.cargar_hogar(anio_t, trimestre_t, COLUMNAS_HOGAR, NUCLEO), pm.cargar_hogar(anio_th, trimestre_th, COLUMNAS_HOGAR, NUCLEO)
    ind_t, ind_th = pm.cargar_individual(anio_t, trimestre_t, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO), pm.cargar_individual(anio_th, trimestre_th, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO)
    pares_h4 = panel.vincular_hogares(hogar_t, hogar_th)
    identidad = panel.identidad_nucleo(ind_t, ind_th, pares_h4, 4)
    pct_identidad = (identidad["identidad_confirmada"] == True).mean()

    anio_t1, trimestre_t1 = panel.sumar_trimestres(anio_t, trimestre_t, 1)
    pares_h1_t = panel.vincular_hogares(hogar_t, pm.cargar_hogar(anio_t1, trimestre_t1, COLUMNAS_HOGAR, NUCLEO))
    anio_th1, trimestre_th1 = panel.sumar_trimestres(anio_th, trimestre_th, 1)
    pares_h1_th = panel.vincular_hogares(hogar_th, pm.cargar_hogar(anio_th1, trimestre_th1, COLUMNAS_HOGAR, NUCLEO))

    claves_h4 = set(zip(pares_h4["CODUSU"], pares_h4["NRO_HOGAR_t"], pares_h4["AGLOMERADO"]))
    claves_h1_t = set(zip(pares_h1_t["CODUSU"], pares_h1_t["NRO_HOGAR_t"], pares_h1_t["AGLOMERADO"]))
    claves_h1_th = set(zip(pares_h1_th["CODUSU"], pares_h1_th["NRO_HOGAR_t"], pares_h1_th["AGLOMERADO"]))
    n_h4 = len(claves_h4)
    return {
        "origen": f"{anio_t}T{trimestre_t}", "n_pares_h4": n_h4,
        "pct_identidad_confirmada": pct_identidad,
        "disenoC_h1t0_solapa_h4": len(claves_h1_t & claves_h4) / n_h4 if n_h4 else float("nan"),
        "disenoB_h1t0p4_solapa_h4": len(claves_h1_th & claves_h4) / n_h4 if n_h4 else float("nan"),
    }


diagnostico_h4 = pd.DataFrame([
    _fila_diagnostico_h4(2012, 1), _fila_diagnostico_h4(2013, 1), _fila_diagnostico_h4(2013, 3),
])
display(diagnostico_h4.style.format({
    "pct_identidad_confirmada": "{:.1%}", "disenoC_h1t0_solapa_h4": "{:.1%}", "disenoB_h1t0p4_solapa_h4": "{:.1%}",
}))

## Registro de la corrida

In [ ]:
duracion_total = time.time() - t0_proceso
n_filas_totales = len(todas_trayectorias) if not todas_trayectorias.empty else 0
pm.registrar_corrida(
    META / "corridas.csv", notebook="04_trayectorias",
    parametros={"TRIMESTRES": TRIMESTRES, "REHACER": REHACER},
    n_trimestres=len(trimestres_origen), n_filas=n_filas_totales,
    duracion_segundos=duracion_total, commit_git=COMMIT_GIT,
)
print(f"corrida registrada -- {duracion_total:.1f}s, {n_filas_totales:,} filas de trayectoria")